# 10.3 - Multi-Tool Orchestration

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook builds every orchestration shape from scratch in plain Python: parallel fan-out, a DAG executor, the ReAct loop, an architecture router, a budgeted+checkpointed conversation, a saga with a circuit breaker, and OTel cost tracing. Each is a small, deterministic, runnable model of a pattern the big frameworks implement under the hood - so you see the mechanism, not the magic.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Installs `nest_asyncio`, the one non-stdlib dependency in this notebook. It lets `asyncio.run()` work inside a notebook that already has a running event loop - needed for the parallel-execution cell coming up. Everything else uses only the Python standard library.

> The `openai` line is commented out - none of the runnable cells call a real API, so no key is required.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install nest_asyncio -q  # noqa
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Pure environment prep. The only install is `nest_asyncio`, a shim that patches asyncio so nested event loops are allowed - which is exactly the situation inside a Jupyter/Colab kernel.

**How the code works - step by step**
- **`!pip install nest_asyncio -q`** - installs the nested-event-loop shim quietly; the `# noqa` marks it as an intentional shell magic.
- **commented `# !pip install openai`** - left off because no cell here makes a live model call.

**In one line:** grab `nest_asyncio` so `asyncio.run()` won't choke in the notebook.

**What you'll see:** (no output - environment prep)

## 1 - Parallel execution: overlap independent calls

The cheapest orchestration win: when tool calls don't depend on each other, fire them all at once instead of one after another. This cell times five fake weather lookups (~50ms each) run sequentially versus run through `asyncio.gather`, and shows the wall-clock collapse from the *sum* of latencies to the *slowest single* latency.

In [ ]:
# PARALLEL vs SEQUENTIAL - independent tool calls should run concurrently.
import asyncio, time
import nest_asyncio; nest_asyncio.apply()   # allow asyncio.run() inside a notebook's running event loop

async def get_weather(city):                     # a stand-in tool with fixed latency
    await asyncio.sleep(0.05)
    return {"city": city, "temp_c": {"Hyderabad":34,"Mumbai":30,"Delhi":38,
                                     "Bangalore":28,"Chennai":33}[city]}

CITIES = ["Hyderabad","Mumbai","Delhi","Bangalore","Chennai"]

async def run_sequential():                      # one at a time: latencies ADD UP
    t = time.perf_counter(); out = []
    for c in CITIES:
        out.append(await get_weather(c))
    return out, (time.perf_counter()-t)*1000

async def run_parallel():                        # all at once: wall-clock = the SLOWEST call
    t = time.perf_counter()
    out = await asyncio.gather(*[get_weather(c) for c in CITIES])
    return list(out), (time.perf_counter()-t)*1000

async def main():
    _, seq_ms = await run_sequential()
    _, par_ms = await run_parallel()
    print(f"  {len(CITIES)} independent calls @ 50ms each:")
    print(f"    sequential ~= {len(CITIES)*50}ms  (5 x 50, latencies add up)")
    print(f"    parallel   ~= {round(par_ms/50)*50}ms   (one round-trip, = the slowest call)")
    print(f"    speedup    ~= {len(CITIES)}x")
asyncio.run(main())

# Output:
#   5 independent calls @ 50ms each:
#     sequential ~= 250ms  (5 x 50, latencies add up)
#     parallel   ~= 50ms   (one round-trip, = the slowest call)
#     speedup    ~= 5x

**Explanation**

A timing harness, not a real tool. `get_weather` is a stand-in that just sleeps 50ms, so the two runners isolate one variable: sequential awaiting adds latencies up, while `asyncio.gather` overlaps them so the total is roughly one round-trip. The speedup equals the number of independent calls.

**How the code works - step by step**
- **`get_weather`** - an async stub that sleeps 50ms then returns a canned temperature; the fixed latency is what makes the comparison clean.
- **`run_sequential`** - `await`s each city in a loop, so the five 50ms waits stack to ~250ms.
- **`run_parallel`** - builds all five coroutines and hands them to `asyncio.gather`, so they run concurrently and finish in ~the slowest call's time.
- **`main`** - runs both, prints the two timings and the ~5x speedup; `asyncio.run(main())` (via `nest_asyncio`) drives it inside the notebook.

**In one line:** loop = latencies add; `gather` = latencies overlap -> wall-clock is the slowest call.

**What you'll see:** Prints that 5 calls at 50ms each cost ~250ms sequential but ~50ms parallel, for a ~5x speedup - the wall-clock equals the single slowest call, not the sum.

## 2 - The task DAG: run each node when its inputs are ready

Sequential chains and conditional branches are the same thing viewed as a dependency graph. This cell defines a DAG where each node lists the nodes it depends on, then runs a tiny executor that repeatedly fires every node whose dependencies are already done - so independent nodes land in the same parallel wave and the number of waves is the critical path.

In [ ]:
# THE TASK DAG - each node names the tools it depends on; run a node when its deps are done.
DAG = {
    "search_courses": [],                                  # independent
    "get_gst_rate":   [],                                  # independent (runs WITH search)
    "price_with_gst": ["search_courses", "get_gst_rate"],  # needs BOTH
    "check_budget":   ["price_with_gst"],                  # conditional gate
    "book":           ["check_budget"],                    # only if budget ok
}

def run_dag(dag):
    done, waves = set(), []
    while len(done) < len(dag):
        ready = [n for n in dag if n not in done and all(d in done for d in dag[n])]
        if not ready:                                      # nothing can advance -> a cycle or a missing dependency
            raise ValueError(f"stuck: cyclic or unsatisfiable deps in {set(dag) - done}")
        waves.append(ready)                                # a wave runs in PARALLEL
        for n in ready: done.add(n)
    return waves

waves = run_dag(DAG)
print("  Execution waves (each wave parallel; waves run in sequence):")
for i, w in enumerate(waves, 1):
    print(f"    wave {i}: {w}")
print(f"  Critical path = {len(waves)} hops (not {len(DAG)} - independent nodes overlap).")

# Output:
#   Execution waves (each wave parallel; waves run in sequence):
#     wave 1: ['search_courses', 'get_gst_rate']
#     wave 2: ['price_with_gst']
#     wave 3: ['check_budget']
#     wave 4: ['book']
#   Critical path = 4 hops (not 5 - independent nodes overlap).

**Explanation**

A scheduler, not a model call. The `DAG` dict is pure data - node -> list of prerequisite nodes - and `run_dag` is the whole executor in a few lines: find ready nodes, run them as a wave, mark done, repeat. It also guards against a graph that can never finish (a cycle or a missing dependency) by detecting a wave with nothing ready and raising.

**How the code works - step by step**
- **`DAG`** - maps each task to its prerequisites: `search_courses` and `get_gst_rate` have none (independent), `price_with_gst` needs both, `check_budget` gates on the price, `book` gates on the budget.
- **`run_dag`** - loops until every node is done; each pass computes `ready` = undone nodes whose deps are all done, appends them as one parallel wave, and marks them done.
- **the stuck guard** - if a pass finds nothing ready but nodes remain, it raises `ValueError` - the DAG equivalent of a step cap, catching cycles or unsatisfiable deps.
- **the print loop** - shows each wave and reports the critical path as the wave count.

**In one line:** repeatedly run every node whose deps are done; the number of waves is your true wall-clock.

**What you'll see:** Prints 4 execution waves - wave 1 runs `search_courses` and `get_gst_rate` together, then `price_with_gst`, `check_budget`, `book` - and reports a critical path of 4 hops, not 5 nodes, because the two independent nodes overlap.

## 3 - The ReAct loop: Thought -> Action -> Observation

When you can't know the plan up front, the agent discovers it: reason about the next step, take one tool call, read the result, repeat - until it's done or a step cap fires. This cell runs a deterministic ReAct loop that prices a course (search -> extract price -> add GST) and shows why the step cap is the one line separating a robust agent from an infinite loop.

In [ ]:
# THE ReAct LOOP - Thought -> Action -> Observation, repeat until done or a step cap fires.
TOOLS = {
    "search":        lambda q: "GenAI Bootcamp INR 14999",
    "extract_price": lambda s: 14999,
    "add_gst":       lambda p: round(int(p)*1.18, 2),
}
def mock_reason(scratch):                         # a deterministic 'model' so the loop reproduces
    if "price"  not in scratch: return ("I need the price", "search", "cost?")
    if "amount" not in scratch: return ("Extract the number", "extract_price", scratch["price"])
    if "total"  not in scratch: return ("Add 18% GST", "add_gst", scratch["amount"])
    return ("I have the answer", "finish", scratch["total"])

def react(max_steps=6):
    scratch = {}
    for s in range(1, max_steps+1):
        thought, action, arg = mock_reason(scratch)
        if action == "finish":
            print(f"  step {s}: THOUGHT: {thought} -> ANSWER: total = {arg}"); return arg
        obs = TOOLS[action](arg)
        print(f"  step {s}: THOUGHT: {thought} | ACTION: {action} | OBS: {obs}")
        scratch |= {"search":{"price":obs}, "extract_price":{"amount":obs},
                    "add_gst":{"total":obs}}[action]
    print(f"  step cap ({max_steps}) hit -> stop; never loop forever"); return None
react()

# Output:
#   step 1: THOUGHT: I need the price | ACTION: search | OBS: GenAI Bootcamp INR 14999
#   step 2: THOUGHT: Extract the number | ACTION: extract_price | OBS: 14999
#   step 3: THOUGHT: Add 18% GST | ACTION: add_gst | OBS: 17698.82
#   step 4: THOUGHT: I have the answer -> ANSWER: total = 17698.82

**Explanation**

A hand-rolled agent loop with the model stubbed out for reproducibility. `mock_reason` plays the model - it inspects the scratchpad and decides the next action deterministically - while `react` is the real machinery: call the chosen tool, record the observation, and bound the whole thing with `max_steps` so it can never spin forever.

**How the code works - step by step**
- **`TOOLS`** - three lambda 'tools': `search` returns a listing, `extract_price` returns the number, `add_gst` multiplies by 1.18.
- **`mock_reason`** - stands in for the LLM; it looks at what the scratchpad is missing and returns the next `(thought, action, arg)`, ending with `finish` once the total exists.
- **`react`** - loops up to `max_steps`: on `finish` it prints the answer and returns; otherwise it runs the tool, prints THOUGHT/ACTION/OBS, and merges the result into `scratch`.
- **the `scratch |= {...}[action]` line** - maps each action's observation into the right scratchpad key (`price`/`amount`/`total`).
- **the step-cap branch** - if the loop exhausts `max_steps`, it prints the cap message and returns `None` instead of looping forever.

**In one line:** reason -> act -> observe -> repeat, bounded by a step cap.

**What you'll see:** Prints 4 steps - search finds the bootcamp, extract_price gets 14999, add_gst yields 17698.82, then the final thought reports total = 17698.82 - all within the step cap, no infinite loop.

## 4 - Agent architectures: start simple, climb only when forced

How many agents does a task need? The guidance is blunt: start with one. This cell demonstrates the two lowest rungs above single-agent - a `router` (a cheap classifier that picks a specialist AND a model tier) and a `planner` (which decomposes a complex goal into an ordered task list) - so you can see the coordination each rung adds.

In [ ]:
# AGENT ARCHITECTURES - start simple; add structure only when one agent cannot cope.
def router(query):                                # a cheap classifier -> specialist + model tier
    q = query.lower()
    if any(w in q for w in ("refund","cancel","order")): return ("orders_agent", "haiku (cheap)")
    if any(w in q for w in ("bug","error","crash")):     return ("eng_agent",    "sonnet (capable)")
    return ("general_agent", "haiku (cheap)")

def planner(goal):                                # decompose a complex goal into an ordered plan
    return ["search options", "compare on price", "check budget", "book or suggest cheaper"]

for q in ["I want a refund for order 42", "the app crashes on login"]:
    agent, tier = router(q)
    print(f"  ROUTER: {q!r:34s} -> {agent:14s} [{tier}]")
print("  PLANNER decomposes 'plan my Goa trip':")
for i, task in enumerate(planner("goa"), 1):
    print(f"    {i}. {task}")

# Output:
#   ROUTER: 'I want a refund for order 42'   -> orders_agent   [haiku (cheap)]
#   ROUTER: 'the app crashes on login'       -> eng_agent      [sonnet (capable)]
#   PLANNER decomposes 'plan my Goa trip':
#     1. search options
#     2. compare on price
#     3. check budget
#     4. book or suggest cheaper

**Explanation**

Two plain functions modeling the architecture ladder - no LLM involved. `router` is keyword matching that returns both which specialist handles a request and which model tier it deserves (Haiku for cheap/easy, Sonnet for hard); `planner` returns a fixed ordered decomposition, standing in for the planner-executor pattern.

**How the code works - step by step**
- **`router`** - lowercases the query and keyword-matches: refund/cancel/order -> `orders_agent` on Haiku, bug/error/crash -> `eng_agent` on Sonnet, else `general_agent` on Haiku; returns `(agent, tier)`.
- **`planner`** - returns a hard-coded 4-step ordered plan, illustrating upfront decomposition of a complex goal.
- **the router demo loop** - runs two sample queries and prints which agent and tier each routes to.
- **the planner demo loop** - enumerates the decomposed steps for a trip goal.
- **orchestrator-workers** - deliberately not coded (mentioned in the walkthrough) - it's the top rung, used only for genuinely parallel open-ended work.

**In one line:** classify the request to a specialist + model tier, or decompose it into an ordered plan - pick the simplest rung that copes.

**What you'll see:** Prints the refund query routed to orders_agent on Haiku and the crash query to eng_agent on Sonnet, then the 4-step planner decomposition (search -> compare -> check budget -> book or suggest cheaper).

## 5 - State management: token budget + checkpoint every step

A multi-step run accumulates state, and short-term (conversation) state must stay under a token budget or it overflows the context window. This cell builds a `Conversation` that compacts the oldest turns into a summary when it exceeds budget, and checkpoints after every turn so a crash resumes rather than restarts.

In [ ]:
# STATE - keep the context window under a token budget; checkpoint after every step.
def tok(msg): return max(1, len(msg)//4)          # ~4 chars per token

class Conversation:
    def __init__(self, budget=40):
        self.turns=[]; self.budget=budget; self.summary=None; self.checkpoint=None
    def total(self):
        return (tok(self.summary) if self.summary else 0) + sum(tok(t) for t in self.turns)
    def add(self, msg):
        self.turns.append(msg)
        if self.total() > self.budget:            # COMPACT: fold oldest turns into a summary
            half = len(self.turns)//2
            self.summary = f"[summary of {half} earlier turns]"; self.turns = self.turns[half:]
        self.checkpoint = {"summary": self.summary, "turns": list(self.turns)}   # persist EVERY step

c = Conversation(budget=40)
for i in range(1, 9):
    c.add(f"user turn number {i} with some content here")
print(f"  after 8 turns: {len(c.turns)} live turns + summary={c.summary!r}")
print(f"  token total = {c.total()} (budget {c.budget}) -> compaction kept us under budget")
restored = Conversation()
restored.summary = c.checkpoint["summary"]; restored.turns = c.checkpoint["turns"]
print(f"  crash + resume -> {len(restored.turns)} turns restored, not 0 (resume, not restart)")

# Output:
#   after 8 turns: 2 live turns + summary='[summary of 2 earlier turns]'
#   token total = 27 (budget 40) -> compaction kept us under budget
#   crash + resume -> 2 turns restored, not 0 (resume, not restart)

**Explanation**

A small stateful class demonstrating two production habits: compaction and checkpointing. `tok` is a crude token estimate (~4 chars each); the class watches its running total and, when it breaches budget, folds the oldest half of the turns into a summary string instead of dropping them - and it snapshots itself after every add so the state survives a crash.

**How the code works - step by step**
- **`tok`** - approximates token count as characters // 4 (min 1).
- **`Conversation.__init__`** - holds live `turns`, a `budget`, an optional `summary`, and a `checkpoint` snapshot.
- **`total`** - sums the summary's tokens (if any) plus all live turns' tokens.
- **`add`** - appends the message; if `total` now exceeds `budget`, it compacts by summarizing the oldest half and keeping the rest; then it saves a `checkpoint` of the current summary+turns after EVERY call.
- **the resume block** - a fresh `Conversation` restores `summary` and `turns` from the checkpoint, proving a crash resumes with turns intact, not from zero.

**In one line:** summarize the oldest turns to stay under budget, and snapshot every step so a crash resumes instead of restarting.

**What you'll see:** After 8 turns it prints 2 live turns plus a summary of 2 earlier turns, a token total of 27 under the budget of 40 (compaction worked), and a resume restoring 2 turns - not 0.

## 6 - Error handling for chains: saga + circuit breaker

This is the part demos skip. When an effectful chain fails halfway - the flight booked, the hotel call 503'd - the world is inconsistent, and retrying double-books. This cell pairs every forward action with a compensating transaction (a saga) to roll back cleanly, and adds a circuit breaker that fails fast after repeated failures instead of hammering a dead tool.

In [ ]:
# SAGA + CIRCUIT BREAKER - an effectful chain that fails mid-way must ROLL BACK, not retry.
LEDGER = []
def book_flight(): LEDGER.append("flight"); return "flight booked"
def book_hotel():  raise RuntimeError("hotel API 503")      # inject a mid-chain failure
def book_car():    LEDGER.append("car");    return "car booked"
COMPENSATE = {"flight": lambda: LEDGER.remove("flight"), "car": lambda: LEDGER.remove("car")}

def saga(steps):
    done = []
    for name, forward in steps:
        try:
            forward(); done.append(name); print(f"    OK   {name}")
        except Exception as e:
            print(f"    FAIL {name}: {e} -> roll back {done[::-1]}")
            for d in reversed(done):
                COMPENSATE[d](); print(f"      compensated {d}")
            return False
    return True

print("  Saga (flight -> hotel -> car):")
saga([("flight", book_flight), ("hotel", book_hotel), ("car", book_car)])
print(f"    final ledger = {LEDGER}   (empty = no half-booked trip)")

class CircuitBreaker:
    def __init__(self, threshold=3):
        self.fails = 0; self.threshold = threshold; self.state = "closed"
    def call(self, fn):
        if self.state == "open": return "SKIPPED (circuit open)"
        try:
            r = fn(); self.fails = 0; return r
        except Exception:
            self.fails += 1
            if self.fails >= self.threshold: self.state = "open"
            return f"fail {self.fails}/{self.threshold}" + (" -> OPEN" if self.state=="open" else "")

cb = CircuitBreaker()
print("  Circuit breaker on a flaky tool:")
for i in range(5):
    print(f"    call {i+1}: {cb.call(book_hotel)}")

# Output:
#   Saga (flight -> hotel -> car):
#     OK   flight
#     FAIL hotel: hotel API 503 -> roll back ['flight']
#       compensated flight
#     final ledger = []   (empty = no half-booked trip)
#   Circuit breaker on a flaky tool:
#     call 1: fail 1/3
#     call 2: fail 2/3
#     call 3: fail 3/3 -> OPEN
#     call 4: SKIPPED (circuit open)
#     call 5: SKIPPED (circuit open)

**Explanation**

Two independent failure-handling mechanisms over a shared fake booking ledger. The saga runs forward actions and, on any failure, replays the recorded compensations in reverse to undo completed steps - leaving the ledger empty. The `CircuitBreaker` counts consecutive failures and 'opens' past a threshold, short-circuiting further calls so you stop retrying into the void.

**How the code works - step by step**
- **`book_flight` / `book_hotel` / `book_car`** - forward actions on a global `LEDGER`; `book_hotel` deliberately raises a 503 to inject a mid-chain failure.
- **`COMPENSATE`** - maps each effect to its undo (remove it from the ledger).
- **`saga`** - runs each step, tracking `done`; on an exception it prints the failure and runs `COMPENSATE` for each done step in reverse, then returns `False`.
- **`CircuitBreaker.call`** - if state is `open`, skip immediately; otherwise run the fn, reset the fail count on success, or increment it and flip to `open` once it hits the threshold.
- **the two demos** - the saga run books the flight, fails on the hotel, and rolls the flight back (empty ledger); the breaker demo calls the flaky tool 5 times and opens after 3 failures.

**In one line:** a saga undoes completed steps after a mid-chain failure; a circuit breaker stops hammering a tool that keeps failing.

**What you'll see:** Prints the saga booking the flight, failing on the hotel, compensating the flight (final ledger empty - no half-booked trip), then the circuit breaker failing 1/3, 2/3, 3/3 -> OPEN, and skipping calls 4 and 5.

## 7 - Observability: OTel gen_ai spans + per-tool cost

A single tool call you can eyeball; a twelve-step multi-agent run is invisible without tracing. This cell emits one OpenTelemetry `gen_ai.*` span per tool call, computes each step's dollar cost from token counts and per-model prices, and sums them - so you can point at the one expensive step in a chain.

In [ ]:
# OBSERVABILITY - emit OpenTelemetry gen_ai spans; attribute cost per tool to find the expensive step.
PRICES = {"haiku": (1.0, 5.0), "sonnet": (3.0, 15.0)}     # $ per 1M tokens; illustrative current-tier rates (they move)

def span(tool, model, tin, tout):
    pin, pout = PRICES[model]
    cost = (tin*pin + tout*pout) / 1_000_000
    return {"gen_ai.tool.name": tool, "gen_ai.request.model": model,
            "gen_ai.usage.input_tokens": tin, "gen_ai.usage.output_tokens": tout,
            "cost_usd": round(cost, 6)}

trace = [span("router","haiku",120,20), span("search","haiku",300,150),
         span("price_with_gst","haiku",200,40), span("book","sonnet",400,120)]
print("  Trace (one gen_ai span per tool call):")
for s in trace:
    print(f"    {s['gen_ai.tool.name']:16s} {s['gen_ai.request.model']:7s} "
          f"in={s['gen_ai.usage.input_tokens']:4d} out={s['gen_ai.usage.output_tokens']:4d} "
          f"${s['cost_usd']}")
print(f"  total run cost = ${round(sum(s['cost_usd'] for s in trace), 6)}  (per-tool -> find the pricey step)")

# Output:
#   Trace (one gen_ai span per tool call):
#     router           haiku   in= 120 out=  20 $0.00022
#     search           haiku   in= 300 out= 150 $0.00105
#     price_with_gst   haiku   in= 200 out=  40 $0.0004
#     book             sonnet  in= 400 out= 120 $0.003
#   total run cost = $0.00467  (per-tool -> find the pricey step)

**Explanation**

A tracing/cost harness, not a model call. `span` builds a dict using the OTel GenAI semantic-convention attribute names (`gen_ai.tool.name`, `gen_ai.request.model`, token usage) and attaches a computed `cost_usd`. Assembling a list of spans and summing their costs is exactly how per-tool cost attribution finds the pricey step - here, the Sonnet `book` call.

**How the code works - step by step**
- **`PRICES`** - per-model (input, output) dollars per 1M tokens; illustrative tier rates.
- **`span`** - computes `cost = (tin*pin + tout*pout) / 1e6` and returns a dict keyed by the OTel `gen_ai.*` attribute names plus the rounded cost.
- **`trace`** - a list of four spans (router, search, price_with_gst on Haiku; book on Sonnet) modeling one run.
- **the print loop** - prints each span's tool, model, token counts, and cost, aligned.
- **the total line** - sums `cost_usd` across spans, giving a per-tool-attributed run cost.
- **India note** - the walkthrough adds a rupee column (~x83) and a per-hop translate span (Sarvam/Bhashini) on the same trace.

**In one line:** one gen_ai span per tool call, cost computed per span, summed - so the expensive step is obvious.

**What you'll see:** Prints a 4-span trace with each tool's model, token counts, and cost, showing the Sonnet `book` step ($0.003) dominating, and a total run cost of $0.00467 attributed per tool.

Across eight tiny programs you built the full orchestration stack: parallelism collapses independent calls to the slowest one; a DAG unifies parallel/sequential/conditional and its critical path is your real wall-clock; ReAct discovers the plan step by step, bounded by a step cap; architectures are a ladder you climb only when forced; state stays under a token budget and checkpoints every step; effectful chains roll back with a saga and stop hammering dead tools with a circuit breaker; and OTel gen_ai spans make the whole run debuggable and cost-attributable. Every framework in the 2026 landscape (LangGraph, OpenAI Agents SDK, Claude Agent SDK) is these same primitives with durable state bolted on. Next, Lesson 10.4 exposes these tools over MCP so other clients can call them, and Module 11 turns this loop into fully autonomous agents.